In [ ]:
# Clone repository and setup environment
!git clone https://github.com/your-username/Fine-Tuning-Open-Source-LLM.git
%cd Fine-Tuning-Open-Source-LLM

# Install dependencies
%pip install -q transformers==4.35.0 datasets==2.14.5 peft==0.6.0 accelerate==0.24.0 bitsandbytes==0.39.1
%pip install -q torch==2.2.0+cu118 torchvision==0.17.0+cu118 torchaudio==2.2.0 --index-url https://download.pytorch.org/whl/cu118

# Add project root to Python path
import sys
from pathlib import Path
PROJECT_ROOT = str(Path().absolute())
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

# Import our modules
from model.load_base_model import ModelLoader
from data.prepare_dataset import DatasetPreparator
from train.run_lora_finetune import run_training

# Verify CUDA is available
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('CUDA version:', torch.version.cuda)


In [ ]:
# Load model with QLoRA configuration
print("Loading CodeLlama-7b-Instruct with QLoRA configuration...")
model_loader = ModelLoader("configs/lora_config.json")

print("Loading base model with 4-bit quantization and preparing for LoRA...")
model, tokenizer = model_loader.load_model_and_tokenizer()

# Check model memory usage
if torch.cuda.is_available():
    print(f"GPU Memory Used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    print(f"GPU Memory Cached: {torch.cuda.memory_reserved() / 1e9:.2f} GB")


In [ ]:
# Prepare dataset and run fine-tuning
print("Preparing dataset...")
dataset_preparator = DatasetPreparator(tokenizer=tokenizer)
dataset = dataset_preparator.prepare_dataset()
print(f"Dataset prepared with {len(dataset)} examples")

# Start fine-tuning
print("\nStarting fine-tuning process...")
run_training(
    model=model,
    tokenizer=tokenizer,
    dataset=dataset,
    output_dir="outputs/checkpoints"
)

print("\nTraining completed!")
